**[Course Home](../README.md) | [Python Lab](../python/03_birth_death_chains_and_mm1_mminf.ipynb) | [Prev: Ch 02: Poisson Processes](02_poisson_processes_and_memoryless_arrivals.ipynb) | [Next: Ch 04: Renewal Theory & Inspection Paradox](04_the_inspection_paradox_and_renewal_theory.ipynb)**

*Part I: Foundations of Flow & Memoryless Systems — Chapter 03 of 11*

---

# Chapter 3: Continuous-Time Markov Chains — Birth-Death Detailed Balance, $M/M/1$, $M/M/c$, and Erlang-C Waiting Formulas
### *Detailed Balance, The Geometry of Multiserver Queues, Erlang Delay Probabilities, and The Statistical Economics of Server Pooling*

---

## 1. The Core Engineering Problem: Decentralized vs Centralized Capacity

When engineering cloud worker pools, testbed clusters, or database connection brokers, infrastructure teams encounter an architectural fork in the road:

> *Should we partition traffic across multiple dedicated, isolated worker queues (e.g., one queue per team or per machine), or should we pool all servers behind a single centralized dispatcher with a shared FIFO queue?*

Naively, one might think that 4 servers handling 4 incoming streams of traffic will perform identically regardless of whether they are separated or combined. Total hardware capacity is identical; total arrival rate is identical.

Yet in production, partitioning servers into isolated silos produces catastrophic waiting times, while pooling them into a unified queue slashes waiting times by 80% to 90% without adding a single extra CPU core.

This chapter uncovers the mathematical engine behind this phenomenon: **Continuous-Time Markov Chains (CTMCs)**, the **Birth-Death detailed balance equations**, the classic **$M/M/1$** and **$M/M/c$** queueing models, and Agner Krarup Erlang's immortal **Erlang-C formula**.

---

## 2. Continuous-Time Markov Chains & The Principle of Detailed Balance

A queueing system where arrivals are Poisson and service durations are exponential is a **Continuous-Time Markov Chain (CTMC)**. 

Let the state of the system $n \in \{0, 1, 2, \dots\}$ represent the total number of jobs currently in the system (both waiting in queue and actively being served).

A **Birth-Death Process** is a special CTMC where transitions occur only between adjacent states:
* **Birth ($n \to n + 1$)**: A new job arrives. Transition rate is $\lambda_n$.
* **Death ($n \to n - 1$)**: An active job completes service. Transition rate is $\mu_n$.

```
         lambda_0         lambda_1         lambda_2         lambda_n-1        lambda_n
     +------------>   +------------>   +------------>   +------------>   +------------>
     |            |   |            |   |            |   |            |   |            |
   ( 0 )          ( 1 )          ( 2 )          ( 3 )   ...        ( n )        ( n+1 )
     |            |   |            |   |            |   |            |   |            |
     <------------+   <------------+   <------------+   <------------+   <------------+
          mu_1             mu_2             mu_3             mu_n             mu_n+1
```

### The Detailed Balance Equations
In steady state, the system is in probabilistic equilibrium. Consider a vertical boundary separating state $n$ and state $n+1$. 

In order for the probability mass in states $\{0, 1, \dots, n\}$ to remain constant over time, the rate of probability flow from state $n$ to $n+1$ must exactly balance the rate of probability flow from $n+1$ to $n$:

$$
\pi_n \lambda_n = \pi_{n+1} \mu_{n+1}, \quad \text{for all } n \ge 0
$$

This is the condition of **Detailed Balance**.

By recursive substitution:

$$
\pi_1 = \pi_0 \frac{\lambda_0}{\mu_1}
$$

$$
\pi_2 = \pi_1 \frac{\lambda_1}{\mu_2} = \pi_0 \left(\frac{\lambda_0 \lambda_1}{\mu_1 \mu_2}\right)
$$

In general, for any state $n \ge 1$:

$$
\pi_n = \pi_0 \prod_{i=0}^{n-1} \frac{\lambda_i}{\mu_{i+1}}
$$

Because the system must be in some state at all times, the probabilities must sum to 1.0 (the **normalization condition**):

$$
\sum_{n=0}^\infty \pi_n = 1 \implies \pi_0 \left[ 1 + \sum_{n=1}^\infty \prod_{i=0}^{n-1} \frac{\lambda_i}{\mu_{i+1}} \right] = 1
$$

$$
\pi_0 = \left[ 1 + \sum_{n=1}^\infty \prod_{i=0}^{n-1} \frac{\lambda_i}{\mu_{i+1}} \right]^{-1}
$$

A stationary steady-state distribution exists if and only if the infinite sum converges.

---

## 3. The Single-Server Queue: The $M/M/1$ Model

In the classic $M/M/1$ queue:
* Arrivals occur at constant rate: $\lambda_n = \lambda$ for all $n \ge 0$.
* Service occurs at constant rate: $\mu_n = \mu$ for all $n \ge 1$.
* Traffic intensity is $\rho = \frac{\lambda}{\mu}$.

### 1. State Distribution
Applying detailed balance:

$$
\pi_n = \pi_0 \left(\frac{\lambda}{\mu}\right)^n = \pi_0 \rho^n
$$

Normalizing over all states:

$$
\sum_{n=0}^\infty \pi_n = \pi_0 \sum_{n=0}^\infty \rho^n = \pi_0 \left(\frac{1}{1 - \rho}\right) = 1 \implies \pi_0 = 1 - \rho
$$

Therefore, the steady-state probability of having exactly $n$ jobs in the system follows a **Geometric Distribution**:

$$
\pi_n = (1 - \rho)\rho^n, \quad n \in \{0, 1, 2, \dots\}
$$

### 2. Expected System Metrics
The average number of jobs in the system $L$ is:

$$
L = \sum_{n=0}^\infty n \pi_n = (1 - \rho)\sum_{n=0}^\infty n \rho^n = (1 - \rho) \frac{\rho}{(1 - \rho)^2} = \frac{\rho}{1 - \rho}
$$

The average number of jobs actively being served is $L_s = 1 - \pi_0 = \rho$.
The average number of jobs waiting in the queue buffer $L_q$ is:

$$
L_q = L - L_s = \frac{\rho}{1 - \rho} - \rho = \frac{\rho^2}{1 - \rho}
$$

### 3. Waiting Times via Little's Law
Using Little's Law ($W = L / \lambda$ and $W_q = L_q / \lambda$):

$$
W = \frac{L}{\lambda} = \frac{\rho}{\lambda(1 - \rho)} = \frac{1}{\mu(1 - \rho)} = \frac{1}{\mu - \lambda}
$$

$$
W_q = \frac{L_q}{\lambda} = \frac{\rho^2}{\lambda(1 - \rho)} = \frac{\rho}{\mu(1 - \rho)} = \frac{\lambda}{\mu(\mu - \lambda)}
$$

### 4. Tail Backlog Probabilities
The probability that the queue depth is at least $k$ jobs is:

$$
\mathbb{P}(N \ge k) = \sum_{n=k}^\infty \pi_n = (1 - \rho)\sum_{n=k}^\infty \rho^n = (1 - \rho)\frac{\rho^k}{1 - \rho} = \rho^k
$$

This simple formula $\mathbb{P}(N \ge k) = \rho^k$ allows immediate calculation of buffer overflow risks.

---

## 4. The Multi-Server Queue: The $M/M/c$ Model & Erlang-C

Now consider a pool of $c$ identical parallel servers fed by a single shared FIFO queue.

```
                                  +--------+
                           /----> | Server | ---->
                          /       +--------+
                         /        +--------+
Arrivals       +-------+ -------> | Server | ----> Departures
Poisson(lambda)| QUEUE |          +--------+
-------------> |       | ------->    ...
               +-------+ \        +--------+
                          \-----> | Server | ---->
                                  +--------+ (c servers, each rate mu)
```

### Transition Rates:
* Arrival rate is constant: $\lambda_n = \lambda$ for all $n$.
* Service rate depends on how many servers are busy:
  $$
  \mu_n = \begin{cases} n \mu & \text{if } 0 \le n \le c \quad (\text{all } n \text{ jobs in service}) \\ c \mu & \text{if } n \ge c \quad (\text{all } c \text{ servers fully saturated}) \end{cases}
  $$

Define the **offered load** $a$ (in erlangs) and **traffic intensity per server** $\rho$:

$$
a = \frac{\lambda}{\mu}, \quad \rho = \frac{\lambda}{c \mu} = \frac{a}{c}
$$

Stability requires $\rho < 1.0 \iff \lambda < c \mu$.

### 1. State Probabilities
For $n \le c$:

$$
\pi_n = \pi_0 \frac{\lambda^n}{1\mu \cdot 2\mu \dots n\mu} = \pi_0 \frac{a^n}{n!}
$$

For $n \ge c$:

$$
\pi_n = \pi_0 \frac{a^c}{c!} \left(\frac{\lambda}{c\mu}\right)^{n - c} = \pi_0 \frac{a^c}{c!} \rho^{n - c}
$$

### 2. Normalization Constant $\pi_0$
Summing across all states:

$$
\sum_{n=0}^\infty \pi_n = \pi_0 \left[ \sum_{n=0}^{c-1} \frac{a^n}{n!} + \frac{a^c}{c!} \sum_{k=0}^\infty \rho^k \right] = 1
$$

$$
\pi_0 = \left[ \sum_{n=0}^{c-1} \frac{a^n}{n!} + \frac{a^c}{c!(1 - \rho)} \right]^{-1}
$$

### 3. The Erlang-C Formula: Probability of Waiting
The probability that an arriving customer finds all $c$ servers busy and is forced to wait in the queue is denoted $C(c, a)$ (or $P_Q$):

$$
C(c, a) = \sum_{n=c}^\infty \pi_n = \pi_0 \frac{a^c}{c!} \sum_{k=0}^\infty \rho^k = \frac{\frac{a^c}{c!(1 - \rho)}}{\sum_{n=0}^{c-1} \frac{a^n}{n!} + \frac{a^c}{c!(1 - \rho)}}
$$

This is the celebrated **Erlang-C Formula** (Erlang Delay Formula), published by Danish mathematician A. K. Erlang in 1917.

### 4. Expected Queue Metrics for $M/M/c$
The expected number of jobs waiting in the queue buffer is:

$$
L_q = \sum_{n=c}^\infty (n - c) \pi_n = C(c, a) \frac{\rho}{1 - \rho}
$$

Notice that $L_q$ in an $M/M/c$ queue is simply the $M/M/1$ queue backlog $\frac{\rho}{1 - \rho}$ multiplied by the probability of waiting $C(c, a)$!

Using Little's Law, the mean waiting time in queue is:

$$
W_q = \frac{L_q}{\lambda} = \frac{C(c, a)}{c\mu - \lambda} = \frac{C(c, a) \cdot \mathbb{E}[S]}{c(1 - \rho)}
$$

and the total time in the system is:

$$
W = W_q + \mathbb{E}[S] = W_q + \frac{1}{\mu}
$$

---

## 5. The Infinite-Server Limit: The $M/M/\infty$ Queue

What happens if capacity is effectively unconstrained ($c \to \infty$), such as in serverless function execution (managed serverless function runtimes) or a massively provisioned cloud worker pool?

In an $M/M/\infty$ queue:
* Every arriving job immediately finds a free server: $\mu_n = n \mu$ for all $n$.
* No job ever waits in a queue: $W_q = 0$, $L_q = 0$.
* The state probabilities become:
  $$
  \pi_n = \pi_0 \frac{a^n}{n!}
  $$
  Normalizing:
  $$
  \sum_{n=0}^\infty \frac{a^n}{n!} = e^a \implies \pi_0 = e^{-a}
  $$
  Therefore:
  $$
  \pi_n = \frac{a^n e^{-a}}{n!}, \quad n \in \{0, 1, 2, \dots\}
  $$

The steady-state number of concurrent jobs in an $M/M/\infty$ system follows an **exact Poisson distribution** with mean:

$$
L = a = \frac{\lambda}{\mu} = \lambda \mathbb{E}[S]
$$

This is the theoretical baseline for an unconstrained system: total concurrency equals the arrival rate multiplied by the raw service time.

---

## 6. The Statistical Pooling Gain: Why 1 Shared Queue Beats $c$ Silos

We now analyze the mathematical proof of why pooling servers is one of the most powerful optimizations in systems architecture.

### The Experiment:
Suppose we have a fleet of $c = 4$ identical testbed runners. Each runner processes tests at rate $\mu = 1.0$ jobs/min ($\mathbb{E}[S] = 1.0$ min).
Total incoming traffic across the fleet is $\Lambda = 3.2$ jobs/min. Overall utilization is $\rho = \frac{3.2}{4 \times 1.0} = 0.80$ ($80\%$).

We compare two organizational architectures:

```
ARCHITECTURE A: 4 ISOLATED QUEUES             ARCHITECTURE B: 1 POOLED QUEUE
(Each server handles lambda = 0.80)            (Shared queue handles Lambda = 3.2)

lambda=0.8 --> [Q] --> [Server 1]               Lambda=3.2 --> [SHARED QUEUE] --> [Server 1]
lambda=0.8 --> [Q] --> [Server 2]                                             --> [Server 2]
lambda=0.8 --> [Q] --> [Server 3]                                             --> [Server 3]
lambda=0.8 --> [Q] --> [Server 4]                                             --> [Server 4]
```

### Analysis of Architecture A (4 Isolated $M/M/1$ Queues)
Each server is an independent $M/M/1$ queue with $\lambda_i = 0.80, \mu = 1.0, \rho = 0.80$:
* Mean queue wait time:
  $$W_{q, A} = \frac{\rho}{\mu(1 - \rho)} = \frac{0.80}{1.0(1 - 0.80)} = \frac{0.80}{0.20} = \mathbf{4.00 \text{ minutes}}$$
* Mean queue backlog per server: $L_{q, i} = \frac{0.80^2}{1 - 0.80} = 3.2$ jobs.
* Total fleet backlog across all 4 queues:
  $$L_{q, \text{total}} = 4 \times 3.2 = \mathbf{12.8 \text{ jobs}}$$

### Analysis of Architecture B (1 Pooled $M/M/4$ Queue)
Total arrival rate $\Lambda = 3.2$, $c = 4$, $\mu = 1.0$. Offered load $a = \frac{3.2}{1.0} = 3.2$. Server utilization $\rho = \frac{3.2}{4} = 0.80$.

1. Compute normalization constant $\pi_0$:
   $$\sum_{n=0}^3 \frac{3.2^n}{n!} = 1 + 3.2 + \frac{10.24}{2} + \frac{32.768}{6} = 1 + 3.2 + 5.12 + 5.4613 = 14.7813$$
   $$\frac{a^4}{4!(1 - \rho)} = \frac{104.8576}{24 \times 0.20} = \frac{104.8576}{4.80} = 21.8453$$
   $$\pi_0 = \frac{1}{14.7813 + 21.8453} = \frac{1}{36.6267} \approx \mathbf{0.0273}$$

2. Compute Erlang-C waiting probability:
   $$C(4, 3.2) = 21.8453 \times \pi_0 = 21.8453 \times 0.0273 \approx \mathbf{0.5964 \quad (59.6\%)}$$
   In the pooled system, an arriving job has a $40.4\%$ chance of finding a free server immediately!

3. Compute Mean Queue Wait Time:
   $$W_{q, B} = \frac{C(4, 3.2)}{c\mu - \Lambda} = \frac{0.5964}{4(1.0) - 3.2} = \frac{0.5964}{0.80} = \mathbf{0.7455 \text{ minutes}}$$

4. Compute Mean Queue Backlog:
   $$L_{q, B} = \Lambda W_{q, B} = 3.2 \times 0.7455 = \mathbf{2.386 \text{ jobs}}$$

### The Performance Comparison:

| Metric | Architecture A (4 Silos) | Architecture B (1 Pooled Queue) | Improvement Factor |
| :--- | :---: | :---: | :---: |
| **Total Hardware** | 4 servers | 4 servers | 0% change |
| **Fleet Utilization $\rho$** | 80% | 80% | 0% change |
| **Mean Wait Time $W_q$** | **4.00 min** | **0.75 min** | **$5.37\times$ faster (81% reduction)** |
| **Total In-Queue Backlog** | **12.8 jobs** | **2.39 jobs** | **$81\%$ reduction** |

**Why Pooling Wins**:
In Architecture A, Server 1 may be sitting completely idle with 0 jobs while Server 2 has 4 jobs queued. The capacity of Server 1 is wasted while developers wait in Server 2's line. 
In Architecture B, **no server is ever idle if there is a job waiting anywhere in the system**.

---

## 7. Concrete Numerical Walkthrough: Sizing an Enterprise Runner Fleet

Let us size a production hardware-in-the-loop verification fleet using the Erlang-C formula.

### System Specifications:
* Peak pull-request arrival rate: $\lambda = 24$ PRs/hour.
* Mean test duration: $\mathbb{E}[S] = 10$ minutes ($1/6$ hour, $\mu = 6$ PRs/hour per runner).
* Offered load: $a = \frac{\lambda}{\mu} = \frac{24}{6} = \mathbf{4.0 \text{ erlangs}}$.
* Minimum theoretical servers for stability: $c > a = 4$.

We evaluate fleet sizes from $c = 5$ to $c = 8$:

| Runner Count $c$ | Utilization $\rho = 4.0/c$ | Erlang-C $C(c, 4.0)$ | Mean Queue Wait $W_q$ (min) | P95 Queue Wait (min) |
| :---: | :---: | :---: | :---: | :---: |
| **5** | 0.800 (80%) | 0.5541 (55.4%) | 5.54 min | 24.1 min |
| **6** | 0.667 (66.7%) | 0.2849 (28.5%) | 1.42 min | 7.9 min |
| **7** | 0.571 (57.1%) | 0.1296 (13.0%) | 0.43 min | 2.5 min |
| **8** | 0.500 (50.0%) | 0.0504 (5.0%) | 0.13 min | 0.7 min |

### Key Observations:
* Increasing runners from $c = 5$ to $c = 6$ (a $20\%$ capacity increase) reduces mean queue waiting time from $5.54$ minutes to $1.42$ minutes—a **$74\%$ latency reduction**!
* At $c = 7$, 87% of PRs begin execution instantly with zero queue wait.

---

## 8. Hands-On 3-Tier Scaffolding Challenges

### 🟢 Warmup (Tier 1): Buffer Sizing for an $M/M/1$ Microservice
**Scenario**: An authentication service pod is modeled as an $M/M/1$ queue with processing capacity $\mu = 100$ requests/second. The service operates at an average load of $\lambda = 80$ req/s ($\rho = 0.80$).

1. Calculate the probability $\pi_0$ that the pod is completely idle.
2. What is the probability that the pod has more than 10 requests queued or active ($P(N > 10)$)?
3. What is the minimum buffer capacity $B$ required such that the probability of buffer overflow $P(N > B)$ is less than $10^{-4}$ ($0.01\%$)?

<details>
<summary><b>Click here to view the solution...</b></summary>

**Solution**:
1. **Idle Probability**:
   $$\pi_0 = 1 - \rho = 1 - 0.80 = \mathbf{0.20 \quad (20\%)}$$

2. **Probability of Exceeding 10 Requests**:
   $$P(N > 10) = P(N \ge 11) = \rho^{11} = 0.80^{11} \approx \mathbf{0.0859 \quad (8.59\%)}$$

3. **Buffer Sizing for $P(N > B) \le 10^{-4}$**:
   $$P(N > B) = \rho^{B+1} \le 10^{-4}$$
   Take natural logs:
   $$(B + 1) \ln(\rho) \le \ln(10^{-4})$$
   Since $\ln(0.80) \approx -0.22314$ (negative, flip inequality):
   $$B + 1 \ge \frac{-9.21034}{-0.22314} \approx 41.28 \implies B + 1 \ge 42 \implies B \ge 41$$
   The pod requires a buffer of at least **$41$ request slots**.
</details>

---

### 🟡 Diagnostic (Tier 2): Sizing an $M/M/c$ Fleet for Erlang-C SLA Compliance
**Scenario**: A hardware testbed fleet receives pull requests at rate $\lambda = 18$ PRs/hour. Each test run takes an exponentially distributed duration averaging $\mathbb{E}[S] = 15$ minutes ($\mu = 4$ PRs/hour).
* The SLA mandates that the **probability of waiting in queue must be less than 20%** ($C(c, a) \le 0.20$).

1. Compute the offered load $a = \lambda / \mu$ in erlangs.
2. Evaluate Erlang-C $C(c, a)$ for candidate fleet sizes $c = 6$ and $c = 7$.
3. What is the minimum integer number of servers $c$ required to meet the SLA, and what is the resulting mean queue wait time $W_q$?

<details>
<summary><b>Click here to view the solution...</b></summary>

**Solution**:
1. **Offered Load**:
   $$a = \frac{\lambda}{\mu} = \frac{18}{4} = \mathbf{4.5 \text{ erlangs}}$$
   (At least 5 servers are required for stability, since $c > 4.5$).

2. **Evaluating Erlang-C**:
   * **For $c = 6$ ($\rho = 4.5 / 6 = 0.75$)**:
     $$\sum_{n=0}^5 \frac{4.5^n}{n!} = 1 + 4.5 + 10.125 + 15.1875 + 17.0859 + 15.3773 = 63.2757$$
     $$\frac{4.5^6}{6!(1 - 0.75)} = \frac{8303.7656}{720 \times 0.25} = \frac{8303.7656}{180} = 46.1320$$
     $$\pi_0 = \frac{1}{63.2757 + 46.1320} = \frac{1}{109.4077} \approx 0.00914$$
     $$C(6, 4.5) = 46.1320 \times 0.00914 \approx \mathbf{0.4217 \quad (42.2\%)}$$
     Does not meet the 20% target.

   * **For $c = 7$ ($\rho = 4.5 / 7 \approx 0.6429$)**:
     $$\sum_{n=0}^6 \frac{4.5^n}{n!} = 63.2757 + \frac{4.5^6}{720} = 63.2757 + 11.5330 = 74.8087$$
     $$\frac{4.5^7}{7!(1 - 4.5/7)} = \frac{37366.945}{5040 \times (2.5/7)} = \frac{37366.945}{1800} = 20.7594$$
     $$\pi_0 = \frac{1}{74.8087 + 20.7594} = \frac{1}{95.5681} \approx 0.01046$$
     $$C(7, 4.5) = 20.7594 \times 0.01046 \approx \mathbf{0.217 \quad (21.7\%)}$$
     Still slightly above 20%.

   * **For $c = 8$ ($\rho = 4.5 / 8 = 0.5625$)**:
     $$\sum_{n=0}^7 \frac{4.5^n}{n!} = 74.8087 + \frac{4.5^7}{5040} = 74.8087 + 7.4141 = 82.2228$$
     $$\frac{4.5^8}{8!(1 - 0.5625)} = \frac{168151.25}{40320 \times 0.4375} = \frac{168151.25}{17640} = 9.5324$$
     $$\pi_0 = \frac{1}{82.2228 + 9.5324} = \frac{1}{91.7552} \approx 0.010899$$
     $$C(8, 4.5) = 9.5324 \times 0.010899 \approx \mathbf{0.1039 \quad (10.4\%)}$$
     Meets the $\le 20\%$ SLA!

3. **Conclusion**:
   Minimum servers required: **$c = 8$ runners**.
   Resulting mean queue wait:
   $$W_q = \frac{C(8, 4.5)}{8\mu - \lambda} = \frac{0.1039}{8(4) - 18} = \frac{0.1039}{14} \approx 0.00742 \text{ hours} = \mathbf{0.445 \text{ minutes (26.7 seconds)}}$$
</details>

---

### 🔴 Capstone Challenge (Tier 3): Cost-Optimal Auto-Scaling Trade-Off Function
**Scenario**: You are writing an auto-scaler cost function for an on-demand cloud test fleet:
* Cloud runner cost: $C_{\text{server}} = \$2.50$ per runner-hour.
* Developer wait penalty cost: $C_{\text{wait}} = \$60.00$ per developer-hour spent waiting in queue.
* Workload: Arrival rate $\lambda = 30$ PRs/hour, test duration $\mathbb{E}[S] = 12$ minutes ($\mu = 5$ PRs/hour).
* Offered load: $a = 30 / 5 = 6.0$ erlangs. Minimum servers $c \ge 7$.

Total hourly operational cost function:
$$\text{Cost}(c) = c \cdot C_{\text{server}} + \lambda \cdot W_q(c) \cdot C_{\text{wait}}$$

1. Write the closed-form expression for $\text{Cost}(c)$ using Erlang-C $C(c, a)$.
2. Calculate total hourly cost for $c = 7, 8, 9, 10, 11$.
3. Identify the cost-optimal fleet size $c^*$ that minimizes total corporate expenditure.

<details>
<summary><b>Click here to view the solution...</b></summary>

**Solution**:
1. **Total Cost Formula**:
   $$W_q(c) = \frac{C(c, 6.0)}{5c - 30} \text{ hours}$$
   $$\text{Cost}(c) = 2.50 c + 30 \times \left(\frac{C(c, 6.0)}{5c - 30}\right) \times 60.00 = 2.50 c + \frac{1800 \cdot C(c, 6.0)}{5c - 30}$$

2. **Numerical Evaluation ($a = 6.0$)**:
   * **$c = 7$** ($\rho = 6/7 \approx 0.857$, $5c - 30 = 5$):
     $C(7, 6.0) \approx 0.6558$
     Wait Cost: $\frac{1800 \times 0.6558}{5} = \$236.09$
     Server Cost: $7 \times 2.50 = \$17.50$
     $$\text{Total Cost}(7) = \$253.59/\text{hr}$$

   * **$c = 8$** ($\rho = 6/8 = 0.750$, $5c - 30 = 10$):
     $C(8, 6.0) \approx 0.3846$
     Wait Cost: $\frac{1800 \times 0.3846}{10} = \$69.23$
     Server Cost: $8 \times 2.50 = \$20.00$
     $$\text{Total Cost}(8) = \$89.23/\text{hr}$$

   * **$c = 9$** ($\rho = 6/9 = 0.667$, $5c - 30 = 15$):
     $C(9, 6.0) \approx 0.2030$
     Wait Cost: $\frac{1800 \times 0.2030}{15} = \$24.36$
     Server Cost: $9 \times 2.50 = \$22.50$
     $$\text{Total Cost}(9) = \$46.86/\text{hr}$$

   * **$c = 10$** ($\rho = 6/10 = 0.600$, $5c - 30 = 20$):
     $C(10, 6.0) \approx 0.0957$
     Wait Cost: $\frac{1800 \times 0.0957}{20} = \$8.61$
     Server Cost: $10 \times 2.50 = \$25.00$
     $$\text{Total Cost}(10) = \mathbf{\$33.61/\text{hr}}$$

   * **$c = 11$** ($\rho = 6/11 = 0.545$, $5c - 30 = 25$):
     $C(11, 6.0) \approx 0.0406$
     Wait Cost: $\frac{1800 \times 0.0406}{25} = \$2.92$
     Server Cost: $11 \times 2.50 = \$27.50$
     $$\text{Total Cost}(11) = \$30.42/\text{hr}$$

   * **$c = 12$** ($\rho = 6/12 = 0.500$, $5c - 30 = 30$):
     $C(12, 6.0) \approx 0.0156$
     Wait Cost: $\frac{1800 \times 0.0156}{30} = \$0.94$
     Server Cost: $12 \times 2.50 = \$30.00$
     $$\text{Total Cost}(12) = \mathbf{\$30.94/\text{hr}}$$

   * **$c = 13$**:
     Server Cost: $13 \times 2.50 = \$32.50$ (Wait cost $< \$0.30$, Total $> \$32.70$)

3. **Optimal Fleet Size**:
   The minimum total cost occurs at **$c^* = 11$ runners**, with total cost **\$30.42/hour**. 
   Running only $7$ servers to "save hardware costs" actually costs the company **8 times more money** (\$253.59 vs \$30.42) due to wasted engineering salaries in merge queue delays!
</details>

---

**[Course Home](../README.md) | [Python Lab](../python/03_birth_death_chains_and_mm1_mminf.ipynb) | [Prev: Ch 02: Poisson Processes](02_poisson_processes_and_memoryless_arrivals.ipynb) | [Next: Ch 04: Renewal Theory & Inspection Paradox](04_the_inspection_paradox_and_renewal_theory.ipynb)**
